# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
This dataset is accessed using its Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields, following the Croissant schema.

In [ ]:
# List available record sets and their fields

print("All available record sets and fields (by @id):\n")
for rs in metadata.record_sets:
    print(f"Record set: {rs['@id']}")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"  Field: {field['@id']}")
    else:
        print("  [No fields declared]")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references to record sets and fields use their `@id` for clarity and reproducibility.

In [ ]:
# Find record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print(f"Found record sets: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with columns: {df.columns.tolist()}")
    else:
        print(f"No records found for {record_set_id}.")

# For illustration, select the first available record_set
if dataframes:
    selected_record_set = list(dataframes.keys())[0]
    print(f"\nUsing record set: {selected_record_set}\n")
    display(dataframes[selected_record_set].head())
else:
    selected_record_set = None
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing numeric fields, and grouping data. All fields are referenced by their `@id` from the schema.

In [ ]:
# Identify available numeric fields by @id
import numpy as np

if selected_record_set is not None:
    df = dataframes[selected_record_set]
    # Try to auto-detect numeric fields
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields detected: {numeric_fields}")
    if len(numeric_fields) == 0:
        # Try to convert possibly numeric columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Selected numeric field for filtering: {numeric_field_id}\n")
        # Example filter: threshold as 75th percentile
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {np.round(threshold,2)}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group field (@id)
        # Heuristically pick a non-numeric field with few unique values
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() <= 10:
                group_field = col
                break
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} per {group_field}:")
            display(grouped_df)
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record set selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using columns referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set is not None and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[selected_record_set][numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a group_field was found, make a boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=dataframes[selected_record_set])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("Cannot visualize: No numeric field detected.")

## 6. Conclusion
We demonstrated loading, inspecting, and visualizing the FAIR<sup>2</sup> dataset using the `mlcroissant` library, referencing all data entities using their `@id` as specified by the Croissant schema. For further analysis, explore other record sets or fields by their `@id` and extend the notebook accordingly.